**Set Up**

In [0]:
%sql
-- Owner: Nadine
-- Name: 01 - Setup
-- Purpose: Create the Bronze, Silver, and Gold schemas and validate that the source folder contains exactly the six approved Instacart CSV files.
-- Grain: One validation summary row for the Instacart source folder.

CREATE SCHEMA IF NOT EXISTS workspace.instacart_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_silver;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_gold;

WITH expected_files AS (
  SELECT explode(
    array(
      'aisles.csv',
      'departments.csv',
      'order_products__prior.csv',
      'order_products__train.csv',
      'orders.csv',
      'products.csv'
    )
  ) AS file_name
),
actual_files AS (
  SELECT
    regexp_extract(path, '([^/]+)$', 1) AS file_name
  FROM read_files(
    '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/*.csv',
    format => 'binaryFile'
  )
),
file_checks AS (
  SELECT
    (SELECT COUNT(*) FROM actual_files) AS actual_file_count,
    (
      SELECT COUNT(*)
      FROM actual_files a
      LEFT ANTI JOIN expected_files e
        ON a.file_name = e.file_name
    ) AS unexpected_file_count,
    (
      SELECT COUNT(*)
      FROM expected_files e
      LEFT ANTI JOIN actual_files a
        ON e.file_name = a.file_name
    ) AS missing_file_count
)
SELECT
  actual_file_count,
  unexpected_file_count,
  missing_file_count,
  assert_true(
    actual_file_count = 6,
    'source folder must contain exactly six csv files'
  ) AS file_count_check,
  assert_true(
    unexpected_file_count = 0,
    'source folder contains an unexpected csv file'
  ) AS unexpected_file_check,
  assert_true(
    missing_file_count = 0,
    'one or more required source files are missing'
  ) AS missing_file_check
FROM file_checks;

**Bronze per Table**

In [0]:
%sql
-- Owner: Nadine
-- Name: 02 - Bronze Aisles
-- Purpose: Load Instacart aisle records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per aisle, uniquely identified by aisle_id.

CREATE OR REPLACE TABLE workspace.instacart_bronze.aisles
USING DELTA
COMMENT 'bronze copy of the instacart aisles csv'
AS
SELECT
  aisle_id,
  aisle,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/aisles.csv',
  format => 'csv',
  header => true,
  schema => 'aisle_id INT, aisle STRING',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 03 - Bronze Departments
-- Purpose: Load Instacart department records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per department, uniquely identified by department_id.

CREATE TABLE IF NOT EXISTS workspace.instacart_bronze.departments (
  department_id INT,
  department STRING,
  _rescued_data STRING
)
USING DELTA
COMMENT 'bronze copy of the instacart departments csv';

INSERT OVERWRITE TABLE workspace.instacart_bronze.departments
SELECT
  department_id,
  department,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/departments.csv',
  format => 'csv',
  header => true,
  schema => 'department_id INT, department STRING',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 04 - Bronze Products
-- Purpose: Load Instacart product records into the Bronze Delta table using an explicit schema and correct quotation-mark parsing while retaining rescued data.
-- Grain: One row per product, uniquely identified by product_id.

CREATE OR REPLACE TABLE workspace.instacart_bronze.products
USING DELTA
COMMENT 'bronze copy of the instacart products csv'
AS
SELECT
  product_id,
  product_name,
  aisle_id,
  department_id,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/products.csv',
  format => 'csv',
  header => true,
  quote => '"',
  escape => '"',
  schema => 'product_id INT, product_name STRING, aisle_id INT, department_id INT',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 05 - Bronze Orders
-- Purpose: Load Instacart order records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per Instacart order, uniquely identified by order_id.

CREATE TABLE IF NOT EXISTS workspace.instacart_bronze.orders (
  order_id INT,
  user_id INT,
  eval_set STRING,
  order_number INT,
  order_dow INT,
  order_hour_of_day INT,
  days_since_prior_order DOUBLE,
  _rescued_data STRING
)
USING DELTA
COMMENT 'bronze copy of the instacart orders csv';

INSERT OVERWRITE TABLE workspace.instacart_bronze.orders
SELECT
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  order_hour_of_day,
  days_since_prior_order,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/orders.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, user_id INT, eval_set STRING, order_number INT, order_dow INT, order_hour_of_day INT, days_since_prior_order DOUBLE',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 06 - Bronze Order Products Prior
-- Purpose: Load prior order-product CSV records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one prior order, uniquely identified by (order_id, add_to_cart_order).

CREATE TABLE IF NOT EXISTS workspace.instacart_bronze.order_products_prior (
  order_id INT,
  product_id INT,
  add_to_cart_order INT,
  reordered INT,
  _rescued_data STRING
)
USING DELTA
COMMENT 'bronze copy of the instacart prior order products csv';

INSERT OVERWRITE TABLE workspace.instacart_bronze.order_products_prior
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__prior.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 07 - Bronze Order Products Train
-- Purpose: Load train order-product records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one train order, uniquely identified by (order_id, add_to_cart_order).

CREATE TABLE IF NOT EXISTS workspace.instacart_bronze.order_products_train (
  order_id INT,
  product_id INT,
  add_to_cart_order INT,
  reordered INT,
  _rescued_data STRING
)
USING DELTA
COMMENT 'bronze copy of the instacart train order products csv';

INSERT OVERWRITE TABLE workspace.instacart_bronze.order_products_train
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__train.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

**Bronze Validation**

In [0]:
%sql
-- Owner: Nadine
-- Name: 08 - Bronze Validation Diagnostics
-- Purpose: Identify Bronze tables containing null required identifiers or rescued data and report their row counts.
-- Grain: One validation summary row per Bronze table.

SELECT
  'aisles' AS table_name,
  COUNT(*) AS row_count,
  COUNT_IF(aisle_id IS NULL) AS null_required_ids,
  COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM workspace.instacart_bronze.aisles

UNION ALL

SELECT
  'departments',
  COUNT(*),
  COUNT_IF(department_id IS NULL),
  COUNT_IF(_rescued_data IS NOT NULL)
FROM workspace.instacart_bronze.departments

UNION ALL

SELECT
  'products',
  COUNT(*),
  COUNT_IF(
    product_id IS NULL
    OR aisle_id IS NULL
    OR department_id IS NULL
  ),
  COUNT_IF(_rescued_data IS NOT NULL)
FROM workspace.instacart_bronze.products

UNION ALL

SELECT
  'orders',
  COUNT(*),
  COUNT_IF(
    order_id IS NULL
    OR user_id IS NULL
  ),
  COUNT_IF(_rescued_data IS NOT NULL)
FROM workspace.instacart_bronze.orders

UNION ALL

SELECT
  'order_products_prior',
  COUNT(*),
  COUNT_IF(
    order_id IS NULL
    OR product_id IS NULL
    OR add_to_cart_order IS NULL
  ),
  COUNT_IF(_rescued_data IS NOT NULL)
FROM workspace.instacart_bronze.order_products_prior

UNION ALL

SELECT
  'order_products_train',
  COUNT(*),
  COUNT_IF(
    order_id IS NULL
    OR product_id IS NULL
    OR add_to_cart_order IS NULL
  ),
  COUNT_IF(_rescued_data IS NOT NULL)
FROM workspace.instacart_bronze.order_products_train

ORDER BY
  null_required_ids DESC,
  rescued_rows DESC;